In [20]:
import os
from oneladle import embed_entries, create_faiss_index, calculate_distances, FastTextEmbedder, get_best_f1
import fasttext
import faiss
from sklearn.metrics import roc_auc_score, average_precision_score

In [6]:
# Paths to the data files. Here we assume that they are text files with one entry per line.
# You can easily adjust the code if your data is in a different format.
reference_entries_path = 'reference_entries.log'
inference_entries_path = 'inference_entries.log'

# Ground truth labels for the inference entries.
# This is for benchmarking; in a real use case, you won't have these labels.
labels_path = 'inference_labels.txt'

In [12]:
# Read the reference and inference entries, and the labels.
# Omit the newline characters and convert labels to integers.
with open(reference_entries_path, 'r') as f:
    reference_entries = [line.strip() for line in f.readlines()]

with open(inference_entries_path, 'r') as f:
    inference_entries = [line.strip() for line in f.readlines()]

with open(labels_path, 'r') as f:
    labels = [int(line.strip()) for line in f.readlines()]

In [13]:
# Print the number of entries and the gt anomaly rate.
print(f"Number of reference entries: {len(reference_entries)}")
print(f"Number of inference entries: {len(inference_entries)}")
print(f"Ground truth anomaly rate: {sum(labels) / len(labels):.2%}")

Number of reference entries: 3055932
Number of inference entries: 1414048
Ground truth anomaly rate: 7.42%


In [9]:
# Train a fastText model on the reference entries
ft_model = fasttext.train_unsupervised(reference_entries_path,
                                       model='skipgram',
                                       dim=128)

Read 21M words
Number of words:  72788
Number of labels: 0
Progress: 100.0% words/sec/thread:    8003 lr:  0.000000 avg.loss:  0.427083 ETA:   0h 0m 0s


In [10]:
# Create a FastTextEmbedder using the trained model.
ft_embedder = FastTextEmbedder(ft_model)

In [ ]:
# Embed the reference entries.
reference_embeddings = embed_entries([reference_entries],
                                     ft_embedder,
                                     is_reference=True) # Set to true for reference entries.

In [15]:
inference_embeddings = embed_entries([inference_entries],
                                     ft_embedder,
                                     is_reference=False) # Set to false for inference entries.

In [16]:
# Faiss resource – use GPU if available, otherwise CPU
if faiss.get_num_gpus() > 0:
    print(f"GPU detected. Using GPU for Faiss.")
    res = faiss.StandardGpuResources()
else:
    print(f"No GPU detected. Using CPU for Faiss.")
    res = None

# Create a Faiss index for the reference embeddings.
index = create_faiss_index(reference_embeddings,
                           index_type='ivf',  # Use IVF index for faster search
                           nlist=100,  # Number of clusters for IVF
                           gpu_res=res)   # Number of clusters to search

GPU detected. Using GPU for Faiss.
Training the Faiss index on the reference embeddings...
Training completed in 0.65 seconds.
Adding the reference embeddings to the Faiss index...
Adding completed in 0.08 seconds.


In [ ]:
# Calculate the anomaly scores (distances) for the inference entries.
# For faster search, set nprobe to a smaller value (e.g. 1).
# This may slightly reduce accuracy but significantly speed up the search.
distances = calculate_distances(inference_embeddings,
                                index,
                                reference_embeddings,
                                k=4,  # Number of nearest neighbors to retrieve
                                nprobe=10,  # Number of clusters to search in IVF
                                gpu_res=res)  # GPU resources for Faiss

y_pred = distances['dist'].to_numpy()

In [21]:
# Calculate AUROC and AUPRC.
y_true = labels
auroc = roc_auc_score(y_true, y_pred)
auprc = average_precision_score(y_true, y_pred)
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

# Get the best F1 score and corresponding threshold.
best_f1, best_threshold = get_best_f1(y_true, y_pred)
print(f"Best F1 score: {best_f1:.4f} at threshold: {best_threshold:.4f}")

AUROC: 0.9999
AUPRC: 0.9980
Best F1 score: 0.9978 at threshold: 0.0202
